In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import torch
from wsidata import open_wsi
import lazyslide as zs


def slide_embedding(
    image: str,
    target_dir: str | Path | None = None,
    batch_size: int = 256,
    num_workers: int = 16,
    tile_px: int = 448,
    mpp: float = 0.5,
    **kwargs: Any
) -> None:
    """
    Extract slide-level embedding using TITAN encoder and save to disk.

    Tile-level features are saved as .h5ad, slide-level aggregated
    embedding as .pt, both in the target directory.
    """
    wsi = open_wsi(image, attach_thumbnail=False)

    zs.pp.find_tissues(wsi)
    zs.pp.tile_tissues(wsi, tile_px, mpp=mpp, **kwargs)

    zs.tl.feature_extraction(
        wsi,
        "titan",
        batch_size=batch_size,
        num_workers=num_workers,
        amp=True,
        pbar=True,
    )

    zs.tl.feature_aggregation(wsi, "titan", encoder="titan")

    # Tile-level: (n_tiles, 768)
    adata = wsi.tables["titan_tiles"]

    # Slide-level aggregated: (768, )
    slide_embeddings = torch.as_tensor(
        adata.varm["agg_slide"],
        dtype=torch.float32,
    ).squeeze().cpu()

    target_dir = Path(target_dir) if target_dir else Path(image).parent

    stem = Path(image).stem
    target_h5ad = target_dir / f"{stem}.h5ad"
    target_pt = target_dir / f"{stem}.pt"

    target_h5ad.parent.mkdir(parents=True, exist_ok=True)

    adata.write_h5ad(target_h5ad)

    torch.save(slide_embeddings, target_pt)
    print(f"Saved tile-level features to {target_h5ad}")
    print(f"Saved slide-level features to {target_pt}")